In [1]:
from pyspark.sql import SparkSession
from delta import DeltaTable

# Initialize SparkSession with Delta Lake support
spark = SparkSession.builder \
    .appName("Hello Delta Tables") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/delta-tables") \
    .getOrCreate()

# Define the path to the Delta table within the delta-tables volume
delta_table_path = "/home/jovyan/delta-tables/hello_delta_table"

# 1. Create a Delta table
data = spark.range(0, 5)  # Create a DataFrame with a range of numbers (0-4)
data.write.format("delta").mode("overwrite").save(delta_table_path)
print("Created Delta Table")

# 2. Read from the Delta table
df = spark.read.format("delta").load(delta_table_path)
print("Reading from Delta Table:")
df.show()

# 3. Update data in the Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)
delta_table.update(
    condition="id % 2 == 0",  # Update even IDs
    set={"id": "id + 100"}    # Add 100 to even IDs
)
print("Updated Delta Table:")
delta_table.toDF().show()

# 4. Delete data from the Delta table
delta_table.delete(condition="id > 100")  # Delete rows where ID is greater than 100
print("Deleted rows from Delta Table:")
delta_table.toDF().show()

# 5. Perform a Merge (Upsert)
new_data = spark.range(3, 8)  # Create new data overlapping with existing data
delta_table.alias("old").merge(
    new_data.alias("new"),
    "old.id = new.id"
).whenMatchedUpdate(set={"id": "new.id"}).whenNotMatchedInsert(values={"id": "new.id"}).execute()

print("After Merge (Upsert):")
delta_table.toDF().show()

# Stop the Spark session
spark.stop()


:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-571b0b88-2a91-4bcf-87ae-118a7119519a;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-core_2.12/2.4.0/delta-core_2.12-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-core_2.12;2.4.0!delta-core_2.12.jar (1238ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/2.4.0/delta-storage-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;2.4.0!delta-storage.jar (64ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (111ms)
:: resolution report :: resolve 2460ms :: artif

Created Delta Table
Reading from Delta Table:


+---+
| id|
+---+
|  3|
|  4|
|  0|
|  2|
|  1|
+---+



Updated Delta Table:


+---+
| id|
+---+
|  3|
|104|
|  1|
|102|
|100|
+---+



Deleted rows from Delta Table:


+---+
| id|
+---+
|  3|
|  1|
|100|
+---+



After Merge (Upsert):


+---+
| id|
+---+
|  3|
|  4|
|  5|
|  6|
|  7|
|  1|
|100|
+---+

